# Lesson 6: ➕ Additional variables 

### 🎯 Learning Objectives
In this lesson, we will learn how to derive a few additional oceanographic variables from temperature, salinity, and pressure data. Specifically, these variables are **mixed layer depth**, **potential density anomaly**, **spiciness**, and **oxygen saturation concentration**, and they help interpret the time series of the biogeochemical variables.

### 🛠️ Prerequisites
Before starting this lesson, make sure that you have completed **Lesson 5**.

### ❓ How to Use This Notebook
* 📚 **Read** the tutorial text blocks carefully, as they provide the essential background information behind the code.
* ▶️ **Run** each code cell sequentially by clicking the cell and pressing `Shift + Enter`.
* 📝 **Exercise** your knowledge! At the end of this notebook, we provide active learning exercises where you will need to write the code yourself.

### 🚀 Ready? Let's Get Started!
---

## 📚 Tutorials

### Import libraries

▶️ Run the cell below to import relevant Python libraries for this lesson. In particular, we will use the library `gsw` ([GSW-Python of TEOS-10](https://teos-10.github.io/GSW-Python/gsw_flat.html)) for deriving the four additional variables.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os
import gsw

### Load the <font color="red">interpolated</font> float time series

▶️ Run the cell below to load the interpolated dataset generated in Lesson 5.

In [ ]:
wmoid = 5906513

in_file = f"../floats/{wmoid}/bgc-argo-tutorials_{wmoid}.nc"
ds = xr.open_dataset(in_file) 
ds

### The function `derive_and_add`

▶️ Run the cell below to define a fuction that derives the following four additional variables from temperature, salinity, and pressure, and adds them to the dataset:

* **MLD**: Mixed layer depth (m)
* **SIGMA0**: Potential density anomaly referenced to a 0-bar pressure (kg/m^3 - 1000)
* **SPICINESS0**: Spiciness referenced to a 0-bar pressure (-)
* **O2SOL**: Oxygen saturation concentration (umol/kg)

In [ ]:
def derive_and_add(ds):
    """
    This function derives the four additional variables from T, S, and P.

    INPUT:
    * ds: the interpolated xarray dataset of float time series

    OUTPUT:
    * ds: the updated xarray dataset including the derived variables
    """

    # Calculate only if valid value exists for T and S
    if np.any(np.isfinite(ds['TEMP_ADJUSTED'])) and np.any(np.isfinite(ds['PSAL_ADJUSTED'])):
        
        # Calculate absolute salinity
        SA = gsw.SA_from_SP(ds['PSAL_ADJUSTED'], ds['PRES_ADJUSTED'], ds["LONGITUDE"], ds["LATITUDE"])        
        
        # Calculate conservative temperature
        CT = gsw.CT_from_t(SA, ds['TEMP_ADJUSTED'], ds['PRES_ADJUSTED'])
        
        # Calcualte saturation oxygen concentration (umol/kg)
        ds["O2SOL"] = gsw.O2sol(SA,CT,ds['PRES_ADJUSTED'],ds["LONGITUDE"],ds["LATITUDE"])
        ds["O2SOL"].attrs = {
            'long_name': 'Oxygen saturation concentration',
            'standard_name': 'moles_of_oxygen_per_unit_mass_in_sea_water',
            'comment': 'see O2sol in https://teos-10.github.io/GSW-Python/gsw_flat.html',
            'units': 'micromole/kg',
            'valid_min': np.float32(-5.0),
            'valid_max': np.float32(600.0)
        }
        
        # Spiciness referenced to a pressure level of 0 dbar, i.e. at surface (kg/m^3)
        ds["SPICINESS0"] = gsw.spiciness0(SA,CT)
        ds["SPICINESS0"].attrs = {
            'long_name': 'Spiciness referenced to a pressure of 0 dbar',
            'standard_name': 'spiciness',
            'comment': 'see spiciness0 in https://teos-10.github.io/GSW-Python/gsw_flat.html',
            'units': 'kg m-3',
            'valid_min': np.float32(-100.0),
            'valid_max': np.float32(100.0)
        } 
        
        # Potential Density
        ds["SIGMA0"] = gsw.sigma0(SA, CT)
        ds["SIGMA0"].attrs = {
            'long_name': 'Potential density anomaly (sigma-0)',
            'standard_name': 'sea_water_sigma_theta',
            'comment': 'Potential density anomaly referenced to 0 dbar. See TEOS-10 for details.',
            'units': 'kg/m^3 - 1000',  # or '1' (dimensionless, but often reported as "kg/m3 - 1000")
            'valid_min': np.float32(20.0),
            'valid_max': np.float32(30.0)
        }
        
        # Obtain sigma0 at 10 dbar based on linear interpolation
        sigma0_10 = ds["SIGMA0"].interp(depth=10)
        
        exceed_sigma0 = ds["SIGMA0"] > sigma0_10 + 0.03

        # Find the integer index of the FIRST True value along the depth dimension
        # (In boolean arrays, False is 0 and True is 1. argmax finds the first 1!)
        first_exceed_idx = exceed_sigma0.argmax(dim="depth")

        # Use that index to grab the actual pressure/depth value
        mld_raw = ds["PRES_ADJUSTED"].isel(depth=first_exceed_idx)
        
        # SAFETY CHECK: What if the threshold was NEVER exceeded? 
        # (e.g., extremely deep winter mixing). argmax defaults to 0 in this case.
        # We use .where() to replace those false 0s with NaNs.
        ds["MLD"] = mld_raw.where(exceed_sigma0.any(dim="depth"))
        ds["MLD"].attrs = {
            'long_name': 'Mixed layer depth based on 0.03 kg m-3 density criterion',
            'standard_name': 'mixed_layer_depth',
            'units': 'm',
            'valid_min': np.float32(0.0),
            'valid_max': np.float32(1000.0)
        }        
    
    # This should not happen but just in case...
    else:
        print('All values are NaNs for both TEMP_ADJUSTED AND PSAL_ADJUSTED, so exitting...')

    return ds

▶️ Run the cell below to use the function.

In [ ]:
ds = derive_and_add(ds)
ds

☝️ Do you see that the four variables have been added successfully?

▶️ Run the cell below to visualize the additional variables.

In [ ]:
plt.close('all')

var_list = ["MLD", "SIGMA0", "SPICINESS0", "O2SOL"]

for i in range(len(var_list)):
    plt.figure()
    mmin = ds[var_list[i]].min().astype(float).values
    mmax = ds[var_list[i]].max().astype(float).values
    if ds[var_list[i]].ndim == 2:
        ds[var_list[i]].plot(x="time",vmin=mmin,vmax=mmax)
    elif ds[var_list[i]].ndim == 1:
        ds[var_list[i]].plot(x="time")
    else:
        print('ndim is other than 1 or 2. you should not see this error...')
    plt.title(f"min: {mmin}, max: {mmax}")            
    plt.gca().invert_yaxis()
    plt.tight_layout()

☝️ Do you see the four figures? From the first figure, we can see that the winter convective mixing was stronger in the second year (MLD > 150 m on some days) compared to the others. Do you notice anything interesting from the other figures?

▶️ Run the cell below to save the output by overwriting the interpolated dataset, if you are happy with the output.

In [ ]:
# Save the output temporarily
ds.to_netcdf('tmp.nc')

# Replace the original interpolated dataset with the updated one.
os.replace('tmp.nc',in_file)

**This is the end of the tutorials for this lesson. Hope you enjoyed it!**

---

## 📝 Exercises

### Exercise 1: Derive the additional variables for the float WMO 5905531.

📝 Write the code yourself below and ▶️ run it!

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python

wmoid = 5905531
in_file = f"../floats/{wmoid}/bgc-argo-tutorials_{wmoid}.nc"
ds = xr.open_dataset(in_file)
ds = derive_and_add(ds)

plt.close('all')

var_list = ["MLD", "SIGMA0", "SPICINESS0", "O2SOL"]

for i in range(len(var_list)):
    plt.figure()
    mmin = ds[var_list[i]].min().astype(float).values
    mmax = ds[var_list[i]].max().astype(float).values
    if ds[var_list[i]].ndim == 2:
        ds[var_list[i]].plot(x="time",vmin=mmin,vmax=mmax)
    elif ds[var_list[i]].ndim == 1:
        ds[var_list[i]].plot(x="time")
    else:
        print('ndim is other than 1 or 2. you should not see this error...')
    plt.title(f"min: {mmin}, max: {mmax}")            
    plt.gca().invert_yaxis()
    plt.tight_layout()

# Saving

# Save the output temporarily
ds.to_netcdf('tmp.nc')

# Replace the original interpolated dataset withe the updated one.
os.replace('tmp.nc',in_file) 


---

This is the end of the lesson. If you are using **Binder**, don't forget to dowload the files created in this lesson before you lose connection!

Well done 🎉, take a break 💤, have another cup ☕, and move to the next lesson ✍️ when you are ready 💪

While your memory is fresh, please feel free to provide your user experience on this lesson by visiting [this link](https://forms.gle/oAGmz5RTW4Pp46bt7). Thanks!